### Linear Regression

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error, mean_squared_error
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import (
    LabelEncoder, minmax_scale, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler, TargetEncoder)

import warnings

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)

In [ ]:
df = pd.read_csv(DATA / "renewals_day3.csv",
                 parse_dates=["renewal_date", "customer_since"])

ren = df[df["renewal_status"] == "Renewed"].copy()
ren["change_pct"] = ren["renewal_acv"] / ren["annual_contract_value"] - 1

In [ ]:
def add_features(data):
    data = data.copy()
    data["tenure_years"] = (data["renewal_date"] -
                            data["customer_since"]).dt.days / 365.25
    data["log_acv"] = np.log(data["annual_contract_value"])
    data["acv_per_seat"] = data["annual_contract_value"] / \
        data["seats_purchased"]

    return data

In [ ]:
ren = add_features(ren)
train, test = train_test_split(ren, test_size=0.2, random_state=42)

In [ ]:
NUM_RAW = ["annual_contract_value", "seats_purchased", "discount_pct", "avg_seat_utilization", "utilization_trend",
           "avg_feature_adoption", "support_cases_12m", "nps_score", "employee_count", "term_months"]
NUM = ["log_acv", "discount_pct", "avg_seat_utilization", "utilization_trend",
       "avg_feature_adoption", "support_cases_12m", "nps_score", "term_months", "tenure_years", "acv_per_seat"]
CATEGORICAL = ["segment", "region", "product", "sales_channel", "industry"]
ORDINAL = ["tier"]

In [ ]:
def make_preprocess(numeric):
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(
            handle_unknown="ignore", sparse_output=False)), CATEGORICAL),
        ("ord", OrdinalEncoder(categories=[
         ["Essentials", "Advantage", "Premier"]]), ORDINAL)
    ])


def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "MAE $": mean_absolute_error(y_true, y_pred),
        "RMSE $": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2 $": r2_score(y_true, y_pred),
        "MAPE %": mean_absolute_percentage_error(y_true, y_pred) * 100
    }


def show(rows):
    t = pd.DataFrame(rows)

    print(t.to_string(index=False))

In [ ]:
y_test = test["renewal_acv"]

In [ ]:
rows = [
    evaluate("Renews at Todays' ACV:", y_test,
             test["annual_contract_value"]),
    evaluate("Todays ACV + 3% uplift : ", y_test,
             test["annual_contract_value"] * 1.03)
]

show(rows)

#### Simple linear regression: `change_pct` ~ `utilization_trend`

One feature, one line: `change_pct = b0 + b1 * utilization_trend`.
Least squares picks the `b0`, `b1` that minimise the sum of squared vertical gaps (residuals) between each point and the line.

In [ ]:
X_COL, Y_COL = "utilization_trend", "change_pct"

simple = LinearRegression().fit(train[[X_COL]], train[Y_COL])
b0, b1 = simple.intercept_, simple.coef_[0]

print(f"Equation : change_pct = {b0:.4f} + {b1:.4f} * utilization_trend")
print(f"           (intercept = expected change when trend is flat, i.e. trend = 0 -> {b0:+.1%})\n")

rows = []
for name, part in [("train", train), ("test", test)]:
    pred = simple.predict(part[[X_COL]])
    rows.append({
        "split": name,
        "n": len(part),
        "R2": r2_score(part[Y_COL], pred),
        "MAE (pp)": mean_absolute_error(part[Y_COL], pred) * 100,
        "RMSE (pp)": mean_squared_error(part[Y_COL], pred) ** 0.5 * 100,
    })
show(rows)

r2_train, r2_test = rows[0]["R2"], rows[1]["R2"]
print(f"\nR2 test = {r2_test:.3f} -> utilization_trend alone explains ~{r2_test:.0%} of the variance "
      f"in change_pct on unseen accounts (train {r2_train:.3f}, gap {r2_train - r2_test:+.3f}).")

step = 0.10
print(f"\nWhat a +{step:.2f} change in utilization_trend means:")
print(f"  slope * {step:.2f} = {b1:.4f} * {step:.2f} = {b1 * step:+.4f}"
      f"  ->  renewal change moves by {b1 * step * 100:+.1f} percentage points")
for t in [-0.10, 0.00, 0.10]:
    print(f"  trend = {t:+.2f}  ->  predicted change_pct = {b0 + b1 * t:+.1%}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(train[X_COL], train[Y_COL], s=22, alpha=0.45, color="#2a78d6",
           edgecolor="white", linewidth=0.5, label=f"train (n={len(train)})")
ax.scatter(test[X_COL], test[Y_COL], s=30, alpha=0.9, color="#eb6834",
           edgecolor="white", linewidth=0.5, label=f"test (n={len(test)})", zorder=3)

# Residuals on the test set: least squares picks the line that minimises the sum of these squared vertical gaps
test_pred = simple.predict(test[[X_COL]])
ax.vlines(test[X_COL], test_pred, test[Y_COL], color="#9a9a94", linewidth=0.8, alpha=0.6,
          label="residual (actual - predicted)", zorder=2)

xs = np.linspace(ren[X_COL].min(), ren[X_COL].max(), 100)
ax.plot(xs, b0 + b1 * xs, color="#1f1f1d", linewidth=2,
        label=f"fit: y = {b0:.3f} + {b1:.3f}x", zorder=4)

# Slope triangle: +0.10 in trend -> +b1*0.10 in change_pct
x0 = 0.25  # placed where points are sparse so the triangle is readable
y0 = b0 + b1 * x0
ax.plot([x0, x0 + step], [y0, y0], color="#1f1f1d", linewidth=1.6, linestyle="--", zorder=5)
ax.plot([x0 + step, x0 + step], [y0, y0 + b1 * step], color="#1f1f1d", linewidth=1.6, linestyle="--", zorder=5)
ax.annotate(f"+{step:.2f} trend", (x0 + step / 2, y0), xytext=(0, -14), textcoords="offset points",
            ha="center", fontsize=9, color="#1f1f1d")
ax.annotate(f"{b1 * step * 100:+.1f} pp change", (x0 + step, y0 + b1 * step / 2), xytext=(6, 0),
            textcoords="offset points", va="center", fontsize=9, color="#1f1f1d")

ax.axhline(0, color="#cfcfca", linewidth=0.8, zorder=0)
ax.axvline(0, color="#cfcfca", linewidth=0.8, zorder=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("utilization_trend")
ax.set_ylabel("change_pct  (renewal ACV / current ACV - 1)")
ax.set_title(f"Renewal price change vs utilization trend  |  R2 train {r2_train:.3f}, test {r2_test:.3f}",
             loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.25)
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

#### Using the model on new accounts

- `predict_renewal(model, accounts)` predicts `change_pct` for one account (a dict) or many (a DataFrame), and converts it to a renewal ACV in $. If the rows also carry `renewal_acv` (e.g. held-out rows), the actuals and errors are shown alongside.
- `show_metrics(model)` scores the model on train/test, both on `change_pct` and on renewal ACV in $ next to the "renews at today's ACV" baseline.

In [ ]:
def predict_renewal(model, accounts):
    """Predict change_pct and renewal ACV for new account(s).

    accounts: a dict (one account) or a DataFrame; needs utilization_trend and annual_contract_value.
    """
    new = pd.DataFrame([accounts]) if isinstance(accounts, dict) else accounts.copy()

    out = pd.DataFrame({
        X_COL: new[X_COL],
        "current_acv": new["annual_contract_value"],
        "pred_change_pct": model.predict(new[[X_COL]]),
    }, index=new.index)
    out["pred_renewal_acv"] = out["current_acv"] * (1 + out["pred_change_pct"])

    if "renewal_acv" in new:
        out["actual_change_pct"] = new["renewal_acv"] / new["annual_contract_value"] - 1
        out["actual_renewal_acv"] = new["renewal_acv"]
        out["error $"] = out["actual_renewal_acv"] - out["pred_renewal_acv"]

    lo, hi = train[X_COL].min(), train[X_COL].max()
    outside = ~out[X_COL].between(lo, hi)
    if outside.any():
        print(f"Warning: {outside.sum()} row(s) have {X_COL} outside the training range "
              f"[{lo:+.3f}, {hi:+.3f}] - the line is extrapolating there.\n")

    pct = "{:+.1%}".format
    usd = "${:,.0f}".format
    print(out.to_string(formatters={
        X_COL: "{:+.3f}".format, "current_acv": usd, "pred_change_pct": pct, "pred_renewal_acv": usd,
        "actual_change_pct": pct, "actual_renewal_acv": usd,
        "error $": lambda v: f"{'+' if v >= 0 else '-'}${abs(v):,.0f}",
    }))
    return out


def show_metrics(model, splits=None):
    """Score the model on change_pct and on renewal ACV in $ (with the today's-ACV baseline for comparison)."""
    splits = splits or {"train": train, "test": test}

    change_rows, dollar_rows = [], []
    for name, part in splits.items():
        pred_change = model.predict(part[[X_COL]])
        change_rows.append({
            "split": name,
            "n": len(part),
            "R2": r2_score(part[Y_COL], pred_change),
            "MAE (pp)": mean_absolute_error(part[Y_COL], pred_change) * 100,
            "RMSE (pp)": mean_squared_error(part[Y_COL], pred_change) ** 0.5 * 100,
        })
        pred_acv = part["annual_contract_value"] * (1 + pred_change)
        dollar_rows.append(evaluate(f"trend model ({name})", part["renewal_acv"], pred_acv))

    last = list(splits.values())[-1]
    dollar_rows.append(evaluate("baseline: today's ACV", last["renewal_acv"], last["annual_contract_value"]))

    print("On change_pct (what the model predicts):")
    show(change_rows)
    print("\nOn renewal ACV in $ (baseline scored on the last split):")
    show(dollar_rows)

In [ ]:
# 1) A brand-new account: only the inputs are known
new_account = {"annual_contract_value": 120_000, "utilization_trend": 0.12}
print("New account:")
predict_renewal(simple, new_account)

# 2) A few held-out test accounts: actuals are known, so we can see the error per row
print("\nHeld-out test accounts:")
predict_renewal(simple, test.sample(5, random_state=7))

# 3) Overall metrics
print()
show_metrics(simple)